In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pickle
import tensorflow as tf
import keras
from keras import ops
import random

from sklearn.preprocessing import MinMaxScaler
from robot_vlp.config import INTERIM_DATA_DIR, PROCESSED_DATA_DIR, TRAINING_LOGS_DIR, MODELS_DIR

import robot_vlp.data.preprocessing as p
import robot_vlp.modeling.rnn_hyperparameter_search as hps
import robot_vlp.modeling.rnn as rnn

tf.config.set_visible_devices([], 'GPU')
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
with open(PROCESSED_DATA_DIR/'exp_vive_navigated_data.pkl', 'rb') as handle:
    data = pickle.load(handle)

In [11]:
tuner = hps.build_random_search_tuner(TRAINING_LOGS_DIR, 'exp_vive_rnd_search', overwrite = False)

Reloading Tuner from /Users/tyrelglass/PhD/Repositories/robot-vlp/models/training_logs/exp_vive_rnd_search/tuner0.json


In [12]:
tensorboard_cb = tf.keras.callbacks.TensorBoard(TRAINING_LOGS_DIR / 'exp_vive_rnd_search/tensorboard')
early_stopping_cb = tf.keras.callbacks.EarlyStopping(patience=3)

In [13]:
tuner.search(
    x = data['X_train'],
    y = [data['y_train'][:,[0,1]],  p.ang_to_vector(data['y_train'][:,2], unit = 'degrees').numpy()],
    epochs = 20,
    validation_data = (data['X_valid'][:], [data['y_valid'][:,[0,1]], p.ang_to_vector(data['y_valid'][:,2], unit = 'degrees').numpy()]), 
    callbacks = [early_stopping_cb, tensorboard_cb],
    batch_size = 32
    )

In [6]:
model = tuner.get_best_models()[0]

/Users/tyrelglass/miniforge3/envs/robot-vlp/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:719: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 40 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [31]:
model.save(MODELS_DIR / 'exp_vive_trained_rnn.keras')

In [7]:
model = keras.models.load_model(MODELS_DIR / 'exp_vive_trained_rnn.keras',custom_objects={"ang_loss_fn": rnn.ang_loss_fn})



/Users/tyrelglass/miniforge3/envs/robot-vlp/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:719: UserWarning: Skipping variable loading for optimizer 'adam', because it has 40 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
%load_ext tensorboard
%tensorboard --logdir=./models/training_logs/exp_vive_rnd_search